In [1]:
import numpy as np
import pickle
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import glob
import re
from datetime import datetime, timedelta, timezone
import gzip
import time
import gc

import bb_behavior.io.videos
from bb_binary.parsing import parse_image_fname_basler, parse_video_fname

In [2]:
import sys, os

WORKING_DIRECTORY = "/home/beesbook/jacob/bb_local_sync/MAIN/2025/"
os.chdir(WORKING_DIRECTORY)
%load_ext autoreload
%autoreload 2

sys.path.insert(0, "/home/beesbook/jacob/bb_local_sync/MAIN")

import importlib
import bb_metrics
from bb_metrics.config import berlin2025 as cfg
import bb_metrics.datafunctions as dfunc

bb_metrics.set_config(cfg)
importlib.reload(dfunc)

dfunc.init(cfg)

bd = cfg


# Functions

# Import and settings

In [3]:
basedir = cfg.basedir
datadir = cfg.basedir / "data_alldetections"
metricsdir = cfg.metrics_dir

# get all datafiles and filter
all_datafiles = sorted(Path(datadir).glob("*.parquet"))
no_data_size = 0.000976

# create metadata dataframe
df_datafiles = []
for datafile in all_datafiles:
    cam_id, timestamp_start, timestamp_end = parse_video_fname(str(datafile))
    if cam_id is not None:
        hive = bd.cam_hive_map[cam_id]
        file_size = datafile.stat().st_size / 1e6
        df_datafiles.append(
            {
                "datafile": str(datafile),
                "cam_id": cam_id,
                "timestamp_start": timestamp_start,
                "timestamp_end": timestamp_end,
                "hive": hive,
                "file_size_mb": file_size,
            }
        )

df_datafiles = pd.DataFrame(df_datafiles)
df_datafiles = df_datafiles[df_datafiles["file_size_mb"] > no_data_size]


# Get metrics and info per time segment

### Parallel

In [4]:
# Using bb_metrics.datafunctions.process_timestamp_chunk and get_valid_timestamp_starts.


In [6]:
## run in parallel on one of the recording computers

from multiprocessing import Pool

time_division = "60s"
num_processes = 2
savedir = Path(metricsdir) / "alldetections"

for hive in bd.hives:
    timestamp_starts = dfunc.get_valid_timestamp_starts(hive, df_datafiles)

    timestamp_ends = timestamp_starts + timedelta(hours=1)

    tasks = []
    for ts_start, ts_end in zip(timestamp_starts, timestamp_ends):
        tasks.append((ts_start, ts_end, hive, df_datafiles, savedir, time_division))

    with Pool(processes=num_processes) as pool:
        pool.starmap(dfunc.process_timestamp_chunk, tasks)


[DONE] 2025-07-14 00:00:00+00:00: processed in 1.58 sec => /mnt/trove/beesbook2025/metrics/alldetections/hive_A_ts_2025-07-14 00_00_00+00_00_dt_60s.pklz
[DONE] 2025-06-14 00:00:00+00:00: processed in 5.81 sec => /mnt/trove/beesbook2025/metrics/alldetections/hive_A_ts_2025-06-14 00_00_00+00_00_dt_60s.pklz
[DONE] 2025-08-02 00:00:00+00:00: processed in 3.68 sec => /mnt/trove/beesbook2025/metrics/alldetections/hive_A_ts_2025-08-02 00_00_00+00_00_dt_60s.pklz
[DONE] 2025-06-15 00:00:00+00:00: processed in 7.15 sec => /mnt/trove/beesbook2025/metrics/alldetections/hive_A_ts_2025-06-15 00_00_00+00_00_dt_60s.pklz
[DONE] 2025-08-03 00:00:00+00:00: processed in 6.55 sec => /mnt/trove/beesbook2025/metrics/alldetections/hive_A_ts_2025-08-03 00_00_00+00_00_dt_60s.pklz
[DONE] 2025-08-04 00:00:00+00:00: processed in 5.0 sec => /mnt/trove/beesbook2025/metrics/alldetections/hive_A_ts_2025-08-04 00_00_00+00_00_dt_60s.pklz
[DONE] 2025-08-05 00:00:00+00:00: processed in 0.88 sec => /mnt/trove/beesbook2025/

In [7]:
# --------------------------------------------------------------------
# After parallel processing, load each partial dictionary from disk
# and merge them into dict_detections
# --------------------------------------------------------------------

time_division = "60s"
savedir = Path(metricsdir) / "alldetections"

for hive in bd.hives:
    dict_detections = {}
    timestamp_starts = dfunc.get_valid_timestamp_starts(hive, df_datafiles)
    for ts_start in timestamp_starts:
        outpath = savedir / f"hive_{hive}_ts_{ts_start}_dt_{time_division}.pklz"
        outpath = Path(str(outpath).replace(':', '_'))

        if outpath.exists():
            with gzip.open(outpath, "rb") as f:
                partial_dict = pickle.load(f)
            for seg_start, hive_dict in partial_dict.items():
                if seg_start not in dict_detections:
                    dict_detections[seg_start] = {}
                for h, val in hive_dict.items():
                    dict_detections[seg_start][h] = val

    # -------------------------------------------------


[SAVED] Combined dictionary for hive=A => /mnt/trove/beesbook2025/metrics//hive_A_dict_detections_60s.pklz
[SAVED] Combined dictionary for hive=B => /mnt/trove/beesbook2025/metrics//hive_B_dict_detections_60s.pklz
[SAVED] Combined dictionary for hive=C => /mnt/trove/beesbook2025/metrics//hive_C_dict_detections_60s.pklz
[SAVED] Combined dictionary for hive=D => /mnt/trove/beesbook2025/metrics//hive_D_dict_detections_60s.pklz
